# SDSS Photometric Redshift Regression with KAN
---
This project attempts to replicate the dataset and results of Vanzella's paper using KANs as an alternative to MLPs.
It should be noted that while this code's dataset uses SDSS DR8 (2011) and the paper's dataset uses SDSS DR1 (2003), data handling (such as which inputs and what types of samples) remains identical.
### Vanzella et al. (2004): https://arxiv.org/pdf/astro-ph/0312064


## Setup and Dependencies

In [ ]:
!pip install pykan astroML pandas

In [ ]:
import torch
from kan import KAN
import numpy as np
import json
from pathlib import Path

# Set seed for fixed outputs
import random
random.seed(67)
rng = np.random.default_rng(67)
torch.manual_seed(67)
torch.cuda.manual_seed_all(67)

if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


## Data Preparation

### Download and Process SDSS Data

Attempt to replicate SDSS DR1 using astroML's SDSS DR8.
Add 7 photometric features as described in the paper:
- Colors: u−g, g−r, r−i, i−z
- Magnitude: r
- Morphology: PetR50, PetR90

Target: spectroscopic redshift (z)

In [ ]:
import pandas as pd
from astroML.datasets import fetch_sdss_specgals

def load_sdss_dataframe():
    data = fetch_sdss_specgals()
    df = pd.DataFrame.from_records(data)

    needed = [
        "z",
        "modelMag_u",
        "modelMag_g",
        "modelMag_r",
        "modelMag_i",
        "modelMag_z",
        "petroR50_r",
        "petroR90_r",
    ]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df[needed].copy()
    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    # Build features
    df["u_g"] = df["modelMag_u"] - df["modelMag_g"]
    df["g_r"] = df["modelMag_g"] - df["modelMag_r"]
    df["r_i"] = df["modelMag_r"] - df["modelMag_i"]
    df["i_z"] = df["modelMag_i"] - df["modelMag_z"]
    df["r"] = df["modelMag_r"]
    df["PetR50"] = df["petroR50_r"]
    df["PetR90"] = df["petroR90_r"]

    return df[["u_g", "g_r", "r_i", "i_z", "r", "PetR50", "PetR90", "z"]]

print("Loading SDSS data...")
df = load_sdss_dataframe()
print(f"Loaded {len(df)} galaxies")
print(f"\nFeature statistics:")
print(df.describe())

Loading SDSS data...
Loaded 661598 galaxies

Feature statistics:
                 u_g            g_r            r_i            i_z  \
count  661598.000000  661598.000000  661598.000000  661598.000000   
mean        1.683876       0.879913       0.417554       0.300008   
std         0.402287       0.276143       0.132620       0.154877   
min        -6.396723      -1.050401     -11.414545      -9.231512   
25%         1.366662       0.700566       0.380451       0.257409   
50%         1.739954       0.890438       0.426552       0.318959   
75%         1.950390       1.042652       0.468899       0.356615   
max         5.592768       8.664486       1.797799       9.549888   

                   r         PetR50         PetR90              z  
count  661598.000000  661598.000000  661598.000000  661598.000000  
mean       17.048622       2.550126       6.599375       0.110951  
std         0.723307       1.382365       3.302508       0.054500  
min        11.715573       0.263818      

In [ ]:
# Apply filters from the paper
max_z = 0.4
max_r = 17.5

df_filtered = df[(df["z"] >= 0.0) & (df["z"] <= max_z) & (df["r"] <= max_r)].copy()
df_filtered = df_filtered.reset_index(drop=True)

print(f"After filtering (z <= {max_z}, r <= {max_r}): {len(df_filtered)} galaxies")
print(f"\nRedshift range: [{df_filtered['z'].min():.3f}, {df_filtered['z'].max():.3f}]")

After filtering (z <= 0.4, r <= 17.5): 454190 galaxies

Redshift range: [0.020, 0.400]


In [ ]:
# Standardize features to allow efficient model training
y_std = 0

def standardize(train_df, test_df):
    global y_std
    feature_cols = [c for c in train_df.columns if c != "z"]

    x_mean = train_df[feature_cols].mean()
    x_std = train_df[feature_cols].std(ddof=0).replace(0, 1.0)
    y_mean = float(train_df["z"].mean())
    y_std = float(train_df["z"].std(ddof=0))

    train_std = train_df.copy()
    test_std = test_df.copy()

    train_std[feature_cols] = (train_df[feature_cols] - x_mean) / x_std
    test_std[feature_cols] = (test_df[feature_cols] - x_mean) / x_std
    train_std["z"] = (train_df["z"] - y_mean) / y_std
    test_std["z"] = (test_df["z"] - y_mean) / y_std

    stats = {
        "feature_cols": feature_cols,
        "x_mean": x_mean.to_dict(),
        "x_std": x_std.to_dict(),
        "y_mean": y_mean,
        "y_std": y_std,
    }
    return train_std, test_std, stats

# Split into train/test
perm = rng.permutation(len(df_filtered))

train_size = 24892
test_size = 88108

train_idx = perm[:train_size]
test_idx = perm[train_size:train_size + test_size]

train_df = df_filtered.iloc[np.sort(train_idx)].reset_index(drop=True)
test_df = df_filtered.iloc[np.sort(test_idx)].reset_index(drop=True)

print(f"Train set: {len(train_df)} galaxies")
print(f"Test set: {len(test_df)} galaxies")

train_df, test_df, stats = standardize(train_df, test_df)

print(f"\nStandardization stats:")
print(json.dumps(stats, indent=2, default=float))

Train set: 24892 galaxies
Test set: 88108 galaxies

Standardization stats:
{
  "feature_cols": [
    "u_g",
    "g_r",
    "r_i",
    "i_z",
    "r",
    "PetR50",
    "PetR90"
  ],
  "x_mean": {
    "u_g": 1.6884008646011353,
    "g_r": 0.8613680601119995,
    "r_i": 0.4133591949939728,
    "i_z": 0.3026341199874878,
    "r": 16.753705978393555,
    "PetR50": 2.7613883018493652,
    "PetR90": 7.188803672790527
  },
  "x_std": {
    "u_g": 0.3635595440864563,
    "g_r": 0.24396467208862305,
    "r_i": 0.09940426051616669,
    "i_z": 0.13436885178089142,
    "r": 0.6970611214637756,
    "PetR50": 1.329087495803833,
    "PetR90": 3.3074159622192383
  },
  "y_mean": 0.09946650266647339,
  "y_std": 0.0475904643535614
}


In [ ]:
# Convert to pykan format
feature_cols = [c for c in train_df.columns if c != "z"]

x_train = torch.tensor(train_df[feature_cols].to_numpy(dtype=np.float32), device=device)
y_train = torch.tensor(train_df[["z"]].to_numpy(dtype=np.float32), device=device)
x_test = torch.tensor(test_df[feature_cols].to_numpy(dtype=np.float32), device=device)
y_test = torch.tensor(test_df[["z"]].to_numpy(dtype=np.float32), device=device)

dataset = {
    "train_input": x_train,
    "train_label": y_train,
    "test_input": x_test,
    "test_label": y_test,
}

print(f"Dataset shapes:")
print(f"  train_input:  {dataset['train_input'].shape}")
print(f"  train_label:  {dataset['train_label'].shape}")
print(f"  test_input:   {dataset['test_input'].shape}")
print(f"  test_label:   {dataset['test_label'].shape}")

Dataset shapes:
  train_input:  torch.Size([24892, 7])
  train_label:  torch.Size([24892, 1])
  test_input:   torch.Size([88108, 7])
  test_label:   torch.Size([88108, 1])


## Creating and Training the KAN

In [ ]:
test_mae_list = []
test_rmse_list = []

def train_mse():
        with torch.no_grad():
            predictions = model(dataset['train_input'])
            mse = torch.nn.functional.mse_loss(predictions, dataset['train_label'])
        return mse

def test_mse():
        with torch.no_grad():
            predictions = model(dataset['test_input'])
            mse = torch.nn.functional.mse_loss(predictions, dataset['test_label'])
        return mse

# Train 20 unique KANs
for datum in range(20):
    model = KAN(
        width=[7, 10, 5, 1],
        grid=5,
        k=4,
        seed=datum,
        device=device
    )

    model.speed()

    results = model.fit(
        dataset,
        opt="Adam",
        update_grid=False,
        lr=2.5e-3,
        metrics=(train_mse, test_mse),
        loss_fn=torch.nn.MSELoss(),
        steps=925
    )

    # Standardize back to values comparable to the paper by multiplying
    # MAE and RMSE by y_std, which was calculated earlier
    test_rmse = (test_mse() ** 0.5) * y_std
    x = dataset["test_input"]
    y = dataset["test_label"].view(-1)
    pred = model(x).view(-1)
    test_err = pred - y
    test_mae = (torch.mean(torch.abs(test_err)).item()) * y_std

    test_mae_list.append(test_mae)
    test_rmse_list.append(test_rmse)
    print(f"Appending following values: MAE={test_mae}, RMSE={test_rmse}")


checkpoint directory created: ./model
saving model version 0.0


| train_loss: 3.65e-01 | test_loss: 3.94e-01 | reg: 0.00e+00 | : 100%|█| 925/925 [00:36<00:00, 25.33


Appending following values: MAE=0.01354165158025289, RMSE=0.01876205950975418
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 3.65e-01 | test_loss: 3.98e-01 | reg: 0.00e+00 | : 100%|█| 925/925 [00:36<00:00, 25.58


Appending following values: MAE=0.013635672527246179, RMSE=0.018918495625257492
checkpoint directory created: ./model
saving model version 0.0


...

Appending following values: MAE=0.013633526629721615, RMSE=0.018960459157824516
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 3.65e-01 | test_loss: 3.95e-01 | reg: 0.00e+00 | : 100%|█| 925/925 [00:36<00:00, 25.54


Appending following values: MAE=0.013574669752342672, RMSE=0.018784087151288986
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 3.65e-01 | test_loss: 3.94e-01 | reg: 0.00e+00 | : 100%|█| 925/925 [00:36<00:00, 25.56

Appending following values: MAE=0.013543791804552008, RMSE=0.01874392107129097


## Evaluation Metrics

In [ ]:
# Mean mae
print(f"Mean Test MAE: {np.mean(test_mae_list)}")
# Convert RMSE tensors to CPU and then to Python numbers before calculating mean and std
test_rmse_list_cpu = [rmse.cpu().item() for rmse in test_rmse_list]
print(f"Mean Test RMSE: {np.mean(test_rmse_list_cpu)}")

# Sample standard deviation
print(f"STDV Test MAE: {np.std(test_mae_list, ddof=1)}")
print(f"STDV Test RMSE: {np.std(test_rmse_list_cpu, ddof=1)}")

Mean Test MAE: 0.013585556884717543
Mean Test RMSE: 0.018828308768570425
STDV Test MAE: 5.398238554066788e-05
STDV Test RMSE: 8.713294413963024e-05
